In [1]:
import pandas as pd
from Bio import Entrez
import time

# Define the file path
file_path = "../Datafile Raw/ov_tcga_pan_can_atlas_2018/data_log2_cna.txt"

# Load the data
df = pd.read_csv(file_path, sep="\t", low_memory=False)
df.head()



,Hugo_Symbol,Entrez_Gene_Id,TCGA-04-1331-01,TCGA-04-1332-01,TCGA-04-1335-01,TCGA-04-1336-01,TCGA-04-1337-01,TCGA-04-1338-01,TCGA-04-1341-01,TCGA-04-1342-01,...,TCGA-72-4235-01,TCGA-72-4236-01,TCGA-72-4237-01,TCGA-72-4238-01,TCGA-72-4240-01,TCGA-72-4241-01,TCGA-OY-A56P-01,TCGA-OY-A56Q-01,TCGA-VG-A8LO-01,TCGA-WR-A838-01
0,ACAP3,116983.0,-0.704,0.08,-0.807,0.101,0.021,-0.999,-0.421,0.089,...,-0.076,0.054,0.801,0.04,0.058,0.339,0.816,0.0,0.137,-0.175
1,ACTRT2,140625.0,-0.704,0.08,-0.807,0.101,0.021,-0.999,-0.421,0.089,...,-0.076,0.054,0.801,0.04,0.058,0.339,0.816,0.0,0.137,-0.175
2,AGRN,375790.0,-0.704,0.08,-0.807,0.101,0.021,-0.999,-0.421,0.089,...,-0.076,0.054,0.801,0.04,0.058,0.339,0.816,0.0,0.137,-0.175
3,ANKRD65,441869.0,-0.704,0.08,-0.807,0.101,0.021,-0.999,-0.421,0.089,...,-0.076,0.054,0.801,0.04,0.058,0.339,0.816,0.0,0.137,-0.175
4,ATAD3A,55210.0,-0.704,0.08,-0.807,0.101,0.021,-0.999,-0.421,0.089,...,-0.076,0.054,0.801,0.04,0.058,0.339,0.816,0.0,0.137,-0.175


In [2]:
# Count NaN values in 'Hugo_Symbol'
hugo_nan_count = df['Hugo_Symbol'].isna().sum()

# Count NaN values in 'Entrez_Gene_Id'
entrez_nan_count = df['Entrez_Gene_Id'].isna().sum()

print(f"Number of NaN values in 'Hugo_Symbol': {hugo_nan_count}")
print(f"Number of NaN values in 'Entrez_Gene_Id': {entrez_nan_count}")


Number of NaN values in 'Hugo_Symbol': 0
Number of NaN values in 'Entrez_Gene_Id': 169


In [3]:

# Set your email (required by NCBI Entrez)
Entrez.email = "sshubha1@lakeheadu.ca"

def fetch_entrez_id_from_hugo(hugo_symbol):
    """
    Fetches the Entrez Gene ID for a given Hugo Symbol using the NCBI Entrez API.
    
    Parameters:
    hugo_symbol (str): The Hugo Symbol (gene name) to look up.
    
    Returns:
    str or None: The Entrez Gene ID if found, otherwise None.
    """
    try:
        handle = Entrez.esearch(db="gene", term=f"{hugo_symbol}[Gene Name] AND Homo sapiens[Organism]", retmode="xml")
        records = Entrez.read(handle)

        if records.get("IdList"):
            return records["IdList"][0]  # Return the first Entrez ID found

    except Exception as e:
        print(f"Error fetching Entrez_Gene_Id for Hugo_Symbol {hugo_symbol}: {e}")
    
    return None

def update_missing_entrez_ids(df):
    """
    Updates missing Entrez Gene IDs in the dataframe using Hugo Symbols.

    Parameters:
    df (pd.DataFrame): The dataframe containing 'Hugo_Symbol' and 'Entrez_Gene_Id' columns.

    Returns:
    pd.DataFrame: Updated dataframe with missing Entrez Gene IDs filled.
    """
    print(f"Original shape: {df.shape}")

    # Remove rows where both 'Hugo_Symbol' and 'Entrez_Gene_Id' are missing
    df.dropna(subset=['Hugo_Symbol', 'Entrez_Gene_Id'], how='all', inplace=True)
    print(f"Shape after removing rows where both are missing: {df.shape}")

    # Find rows where 'Entrez_Gene_Id' is missing
    nan_entrez_rows = df[df['Entrez_Gene_Id'].isnull()].copy()

    for idx, row in nan_entrez_rows.iterrows():
        hugo_symbol = row['Hugo_Symbol']

        if pd.notna(hugo_symbol):  # Ensure Hugo_Symbol is valid
            entrez_id = fetch_entrez_id_from_hugo(hugo_symbol)
            if entrez_id:
                df.at[idx, 'Entrez_Gene_Id'] = pd.to_numeric(entrez_id)
                print(f"Updated Entrez_Gene_Id for Hugo_Symbol {hugo_symbol} -> Entrez_Gene_Id {entrez_id}")
            else:
                print(hugo_symbol)

        time.sleep(0.35)  # Respect NCBI's rate limit

    print(f"Shape after filling missing Entrez_Gene_Id: {df.shape}")
    return df

# Usage Example
df = update_missing_entrez_ids(df)


Original shape: (25128, 574)
Shape after removing rows where both are missing: (25128, 574)
Updated Entrez_Gene_Id for Hugo_Symbol hsa-mir-1302-2 -> Entrez_Gene_Id 100302278
hsa-mir-551a
hsa-mir-1273d
Updated Entrez_Gene_Id for Hugo_Symbol hsa-mir-1290 -> Entrez_Gene_Id 100302276
Updated Entrez_Gene_Id for Hugo_Symbol hsa-mir-1256 -> Entrez_Gene_Id 100302155
Updated Entrez_Gene_Id for Hugo_Symbol hsa-mir-552 -> Entrez_Gene_Id 693137
hsa-mir-3116-1
hsa-mir-3116-2
hsa-mir-3118-1
hsa-mir-3118-2
hsa-mir-3118-3
hsa-mir-4259
Updated Entrez_Gene_Id for Hugo_Symbol hsa-mir-1255b-2 -> Entrez_Gene_Id 100313835
hsa-mir-3119-1
Updated Entrez_Gene_Id for Hugo_Symbol hsa-mir-664 -> Entrez_Gene_Id 100302234
hsa-mir-548s
Updated Entrez_Gene_Id for Hugo_Symbol hsa-mir-1285-2 -> Entrez_Gene_Id 100302268
Updated Entrez_Gene_Id for Hugo_Symbol hsa-mir-1302-3 -> Entrez_Gene_Id 100302128
Updated Entrez_Gene_Id for Hugo_Symbol hsa-mir-1245 -> Entrez_Gene_Id 100302219
hsa-mir-3130-1
hsa-mir-3130-3
Updated Ent

In [4]:
# Count NaN values in 'Hugo_Symbol'
hugo_nan_count = df['Hugo_Symbol'].isna().sum()

# Count NaN values in 'Entrez_Gene_Id'
entrez_nan_count = df['Entrez_Gene_Id'].isna().sum()

print(f"Number of NaN values in 'Hugo_Symbol': {hugo_nan_count}")
print(f"Number of NaN values in 'Entrez_Gene_Id': {entrez_nan_count}")


Number of NaN values in 'Hugo_Symbol': 0
Number of NaN values in 'Entrez_Gene_Id': 63


In [5]:
df

,Hugo_Symbol,Entrez_Gene_Id,TCGA-04-1331-01,TCGA-04-1332-01,TCGA-04-1335-01,TCGA-04-1336-01,TCGA-04-1337-01,TCGA-04-1338-01,TCGA-04-1341-01,TCGA-04-1342-01,...,TCGA-72-4235-01,TCGA-72-4236-01,TCGA-72-4237-01,TCGA-72-4238-01,TCGA-72-4240-01,TCGA-72-4241-01,TCGA-OY-A56P-01,TCGA-OY-A56Q-01,TCGA-VG-A8LO-01,TCGA-WR-A838-01
0,ACAP3,116983.0,-0.704,0.080,-0.807,0.101,0.021,-0.999,-0.421,0.089,...,-0.076,0.054,0.801,0.04,0.058,0.339,0.816,0.000,0.137,-0.175
1,ACTRT2,140625.0,-0.704,0.080,-0.807,0.101,0.021,-0.999,-0.421,0.089,...,-0.076,0.054,0.801,0.04,0.058,0.339,0.816,0.000,0.137,-0.175
2,AGRN,375790.0,-0.704,0.080,-0.807,0.101,0.021,-0.999,-0.421,0.089,...,-0.076,0.054,0.801,0.04,0.058,0.339,0.816,0.000,0.137,-0.175
3,ANKRD65,441869.0,-0.704,0.080,-0.807,0.101,0.021,-0.999,-0.421,0.089,...,-0.076,0.054,0.801,0.04,0.058,0.339,0.816,0.000,0.137,-0.175
4,ATAD3A,55210.0,-0.704,0.080,-0.807,0.101,0.021,-0.999,-0.421,0.089,...,-0.076,0.054,0.801,0.04,0.058,0.339,0.816,0.000,0.137,-0.175
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25123,hsa-mir-513c,100302114.0,-0.368,-0.256,0.071,-0.335,0.062,-0.082,-0.589,0.136,...,-0.081,-0.216,-0.146,-0.21,-0.168,-0.100,-0.086,-0.195,-0.675,0.065
25124,hsa-mir-513b,100302253.0,-0.368,-0.256,0.071,-0.335,0.062,-0.082,-0.589,0.136,...,-0.081,-0.216,-0.146,-0.21,-0.168,-0.100,-0.086,-0.195,-0.675,0.065
25125,hsa-mir-513a-1,NaN,-0.368,-0.256,0.071,-0.335,0.062,-0.082,-0.589,0.136,...,-0.081,-0.216,-0.146,-0.21,-0.168,-0.100,-0.086,-0.195,-0.675,0.065
25126,hsa-mir-513a-2,574510.0,-0.368,-0.256,0.071,-0.335,0.062,-0.082,-0.589,0.136,...,-0.081,-0.216,-0.146,-0.21,-0.168,-0.100,-0.086,-0.195,-0.675,0.065


In [6]:
# Identify patient columns (excluding 'Hugo_Symbol' and 'Entrez_Gene_Id')
patient_columns = df.columns[2:]  # Excluding metadata columns

# Calculate the percentage of NaN values for each gene
nan_percentage = df[patient_columns].isna().sum(axis=1) / len(patient_columns)

# Count the number of genes with 80% or more NaN values
num_genes_with_80_percent_nan = (nan_percentage >= 0.8).sum()

# Calculate the total number of genes in the dataset
total_genes = df.shape[0]

# Compute the percentage of affected genes
percentage_of_genes = (num_genes_with_80_percent_nan / total_genes) * 100

# Print the results
print(f"Total number of genes: {total_genes}")
print(f"Number of genes with 80% or more NaN values: {num_genes_with_80_percent_nan}")
print(f"Percentage of genes with 80% or more NaN values: {percentage_of_genes:.2f}%")


Total number of genes: 25128
Number of genes with 80% or more NaN values: 0
Percentage of genes with 80% or more NaN values: 0.00%


In [7]:
# Identify patient columns (excluding 'Hugo_Symbol' and 'Entrez_Gene_Id')
patient_columns = df.columns[2:]  # Excluding metadata columns

# Calculate the percentage of NaN values for each gene
nan_percentage = df[patient_columns].isna().sum(axis=1) / len(patient_columns)

# Filter out rows where 80% or more of the values are NaN
df = df.loc[nan_percentage < 0.8].reset_index(drop=True)

# Print confirmation message
print(f"Updated dataset: {df.shape[0]} genes remaining after removing {nan_percentage.ge(0.8).sum()} genes.")


Updated dataset: 25128 genes remaining after removing 0 genes.


In [8]:
df=df.transpose()

In [9]:
df

,0,1,2,3,4,5,6,7,8,9,...,25118,25119,25120,25121,25122,25123,25124,25125,25126,25127
Hugo_Symbol,ACAP3,ACTRT2,AGRN,ANKRD65,ATAD3A,ATAD3B,ATAD3C,AURKAIP1,B3GALT6,C1orf159,...,hsa-mir-1321,hsa-mir-361,hsa-mir-548m,hsa-mir-652,hsa-mir-220a,hsa-mir-513c,hsa-mir-513b,hsa-mir-513a-1,hsa-mir-513a-2,hsa-mir-224
Entrez_Gene_Id,116983.0,140625.0,375790.0,441869.0,55210.0,83858.0,219293.0,54998.0,126792.0,54991.0,...,100302171.0,494323.0,100302151.0,724022.0,407005.0,100302114.0,100302253.0,NaN,574510.0,NaN
TCGA-04-1331-01,-0.704,-0.704,-0.704,-0.704,-0.704,-0.704,-0.704,-0.704,-0.704,-0.704,...,0.036,0.036,-0.372,-0.348,-0.348,-0.368,-0.368,-0.368,-0.368,-0.368
TCGA-04-1332-01,0.08,0.08,0.08,0.08,0.08,0.08,0.08,0.08,0.08,0.08,...,-0.237,-0.237,-0.237,-0.237,-0.237,-0.256,-0.256,-0.256,-0.256,-0.256
TCGA-04-1335-01,-0.807,-0.807,-0.807,-0.807,-0.807,-0.807,-0.807,-0.807,-0.807,-0.807,...,0.063,0.063,0.063,0.049,0.071,0.071,0.071,0.071,0.071,0.673
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TCGA-72-4241-01,0.339,0.339,0.339,0.339,0.339,0.339,0.339,0.339,0.339,0.339,...,-0.027,-0.027,-0.52,-0.52,-0.1,-0.1,-0.1,-0.1,-0.1,-0.1
TCGA-OY-A56P-01,0.816,0.816,0.816,0.816,0.816,0.816,0.816,0.816,0.816,0.816,...,-0.056,-0.056,-0.056,-0.056,-0.056,-0.086,-0.086,-0.086,-0.086,-0.078
TCGA-OY-A56Q-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-0.031,-0.031,-0.031,-0.093,-0.093,-0.195,-0.195,-0.195,-0.195,-0.195
TCGA-VG-A8LO-01,0.137,0.137,0.137,0.137,0.137,0.137,0.137,0.137,0.137,0.137,...,0.283,0.283,-0.677,-0.677,-0.686,-0.675,-0.675,-0.675,-0.675,-0.675


In [10]:
# Define the file path
file_path = "../Datafiles Final/Datafiles Cleaned/log2cna.pkl"

# Save using Pickle format (best for mixed data types)
df.to_pickle(file_path)

print(f"File saved successfully at: {file_path}")


File saved successfully at: ../Datafiles Final/Datafiles Cleaned/log2cna.pkl


In [11]:
# Load the Pickle file
df_log2cna = pd.read_pickle(file_path)

# Verify the data
df_log2cna

,0,1,2,3,4,5,6,7,8,9,...,25118,25119,25120,25121,25122,25123,25124,25125,25126,25127
Hugo_Symbol,ACAP3,ACTRT2,AGRN,ANKRD65,ATAD3A,ATAD3B,ATAD3C,AURKAIP1,B3GALT6,C1orf159,...,hsa-mir-1321,hsa-mir-361,hsa-mir-548m,hsa-mir-652,hsa-mir-220a,hsa-mir-513c,hsa-mir-513b,hsa-mir-513a-1,hsa-mir-513a-2,hsa-mir-224
Entrez_Gene_Id,116983.0,140625.0,375790.0,441869.0,55210.0,83858.0,219293.0,54998.0,126792.0,54991.0,...,100302171.0,494323.0,100302151.0,724022.0,407005.0,100302114.0,100302253.0,NaN,574510.0,NaN
TCGA-04-1331-01,-0.704,-0.704,-0.704,-0.704,-0.704,-0.704,-0.704,-0.704,-0.704,-0.704,...,0.036,0.036,-0.372,-0.348,-0.348,-0.368,-0.368,-0.368,-0.368,-0.368
TCGA-04-1332-01,0.08,0.08,0.08,0.08,0.08,0.08,0.08,0.08,0.08,0.08,...,-0.237,-0.237,-0.237,-0.237,-0.237,-0.256,-0.256,-0.256,-0.256,-0.256
TCGA-04-1335-01,-0.807,-0.807,-0.807,-0.807,-0.807,-0.807,-0.807,-0.807,-0.807,-0.807,...,0.063,0.063,0.063,0.049,0.071,0.071,0.071,0.071,0.071,0.673
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TCGA-72-4241-01,0.339,0.339,0.339,0.339,0.339,0.339,0.339,0.339,0.339,0.339,...,-0.027,-0.027,-0.52,-0.52,-0.1,-0.1,-0.1,-0.1,-0.1,-0.1
TCGA-OY-A56P-01,0.816,0.816,0.816,0.816,0.816,0.816,0.816,0.816,0.816,0.816,...,-0.056,-0.056,-0.056,-0.056,-0.056,-0.086,-0.086,-0.086,-0.086,-0.078
TCGA-OY-A56Q-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-0.031,-0.031,-0.031,-0.093,-0.093,-0.195,-0.195,-0.195,-0.195,-0.195
TCGA-VG-A8LO-01,0.137,0.137,0.137,0.137,0.137,0.137,0.137,0.137,0.137,0.137,...,0.283,0.283,-0.677,-0.677,-0.686,-0.675,-0.675,-0.675,-0.675,-0.675


In [12]:
#####################################################################################################